In [ ]:
# ==============================================================================
# PARALLEL DIM: CAR
# ==============================================================================
from notebooks.helpers import IncrementalPipeline, TableConfig, get_latest_batch_id, setup_logger, write_gold_table, safe_count, generate_batch_id
import pandas as pd
from notebooks.helpers.silver_transforms import transform_car_full_pipeline, build_equipment_bridges

logger = setup_logger("parallel_dim_car")

batch_id = generate_batch_id()
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)

dependencies = ["car", "inventory_equipment", "equipment"]

bronze_batch_id = get_latest_batch_id(spark, "inventory")
if not bronze_batch_id:
    raise ValueError("No bronze batch_id found for inventory; run bronze load first.")
logger.info(f"Using bronze batch_id for inventory: {bronze_batch_id}")

config = TableConfig(
    table_name="inventory",
    business_key="inventory_id",
    surrogate_key="car_key",
    watermark_column="last_update",
    scd_type=1,
    gold_table_name="dim_car",
    silver_transform=transform_car_full_pipeline,
    dependencies=dependencies,
)

print("Row Counts (Before):")
print(f"dim_car: {safe_count(spark, 'dim_car')}")
print(f"bridge_equipment_group_equipment: {safe_count(spark, 'bridge_equipment_group_equipment')}")
print(f"bridge_car_equipment: {safe_count(spark, 'bridge_car_equipment')}")
print("\nLatest Watermarks (Before):")
display(spark.table("wheelie.monitoring.watermarks"))

results = pipeline.load_tables([config], force_full=False, bronze_batch_id=bronze_batch_id)

# Rebuild equipment bridges after inventory/equipment load
inventory_equipment_bronze = spark.table("wheelie.bronze.inventory_equipment")
bridge_equipment_group_equipment, bridge_car_equipment = build_equipment_bridges(
    inventory_equipment_bronze
)
write_gold_table(bridge_equipment_group_equipment, "bridge_equipment_group_equipment", mode="overwrite")
write_gold_table(bridge_car_equipment, "bridge_car_equipment", mode="overwrite")

logger.info("equipment bridges rebuilt")
display(pd.DataFrame(results))

print("\nRow Counts (After):")
print(f"dim_car: {safe_count(spark, 'dim_car')}")
print(f"bridge_equipment_group_equipment: {safe_count(spark, 'bridge_equipment_group_equipment')}")
print(f"bridge_car_equipment: {safe_count(spark, 'bridge_car_equipment')}")
print("\nLatest Watermarks (After):")
display(spark.table("wheelie.monitoring.watermarks"))
